### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="tour_travels_churn",
    dataset_year="2021",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/tejashvi14/tour-travels-customer-churn-prediction/",
    download_description="""
kaggle datasets download tejashvi14/tour-travels-customer-churn-prediction --unzip && mkdir -p local-data-warehouse/tour_travels_churn && mv Customertravel.csv local-data-warehouse/tour_travels_churn/
""",
    # References
    academic_reference_bibtex=r"""@misc{Tejashvi2023TourTravelsCustomerChurnPrediction,
  author = {Tejashvi},
  title  = {Tour & Travels Customer Churn Prediction},
  year   = {2023},
  howpublished = {\url{https://www.kaggle.com/datasets/tejashvi14/tour-travels-customer-churn-prediction}},
  note   = {Kaggle dataset}
}
""",
    academic_reference_bibtex_key="Tejashvi2023TourTravelsCustomerChurnPrediction",
    license="CC0: Public Domain",
    data_tags=["IID"],
    curation_comments="""
The source of the data from Kaggle is unknown but the data distributions look reasonable enough to use. But I would also not be surprised if we find out the data is artificially created.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Target",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Target",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "Customertravel.csv")
print("Loaded data shape:", df.shape)

as_cat_type = ["FrequentFlyer", "AnnualIncomeClass","AccountSyncedToSocialMedia", "BookedHotelOrNot", "Target"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (954, 7)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 954
Columns: 7
Use sampling: False (sample size: 954)
Get row duplicates (staged, merged)...
Using top-6 columns for initial filtering: ['Age', 'ServicesOpted', 'FrequentFlyer', 'AnnualIncomeClass', 'AccountSyncedToSocialMedia', 'BookedHotelOrNot']
Rows remaining as candidates after top-6 filter: 738 (of 954)

#### Duplicate Report
Total duplicate rows: 507 (53.14% of dataset)
Duplicate rows ignoring target: 535 (56.08% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Age,FrequentFlyer,AnnualIncomeClass,ServicesOpted,AccountSyncedToSocialMedia,BookedHotelOrNot,Target
0,33,No Record,Low Income,2,Yes,No,1
1,37,No,Middle Income,3,Yes,No,0
2,30,No,Middle Income,2,No,Yes,0
3,30,No,Low Income,2,No,No,0
4,30,No,Low Income,1,Yes,No,0


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,FrequentFlyer,category,0.0,0.0,3.0,"No, Yes, No Record"
1,AnnualIncomeClass,category,0.0,0.0,3.0,"Middle Income, Low Income, High Income"
2,AccountSyncedToSocialMedia,category,0.0,0.0,2.0,"No, Yes"
3,BookedHotelOrNot,category,0.0,0.0,2.0,"No, Yes"
4,Target,category,0.0,0.0,2.0,"0, 1"
5,Age,int64,0.0,0.0,11.0,"30, 37, 34, 31, 28, 29, 36, 27, 35, 38"
6,ServicesOpted,int64,0.0,0.0,6.0,"1, 2, 3, 4, 5, 6"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Age,954.0,32.109015,3.337388,27.0,38.0
ServicesOpted,954.0,2.437107,1.606233,1.0,6.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                     rank                             
AccountSyncedToSocialMedia 1                No    594  62.26
                           2               Yes    360  37.74
AnnualIncomeClass          1     Middle Income    409  42.87
                           2        Low Income    386  40.46
                           3       High Income    159  16.67
BookedHotelOrNot           1                No    576  60.38
                           2               Yes    378  39.62
FrequentFlyer              1                No    608  63.73
                           2               Yes    286  29.98
                           3         No Record     60   6.29
Target                     1                 0    730  76.52
                           2                 1    224  23.48

In [8]:
# Target Distribution
target_df

,count,pct
Target,,
0,730,76.52
1,224,23.48


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to tour_travels_churn/019d5dca-742e-7a15-843d-2615543715bb
019d5dca-742e-7a15-843d-2615543715bb
bb003e039775dbef89698996da3ae3f0e68514eb9aa51f91ec02522105138148
